"""
# Bonsai 2 on TPU: single-scan step (notebook 08)

Notebook 06 measured 3.5 tok/s aggregate and profiled as 95%
overhead-bound (Python loop + ~1000 XLA launches per step, not traffic).
This notebook removes the overhead with `lax.scan`:

- All 64 layers run inside ONE compiled program (a `While`, not unrolled),
  so per-step dispatches drop from ~200 to ~5. Python never runs mid-step.
- Weights decode inside the scan from stacked raw blobs (no 108 GB resident,
  no per-layer Python decode). Small weights are stacked per-name.
- The layer math is notebook 06's proven units, called unchanged, so the
  gate (scan vs 06, bit-exact) is strong. fp32 throughout; int8 comes next
  only if this does not reach 100 tok/s aggregate on its own.
- Batching is unchanged from 06 (vmap'd units, same prompt replicated).

Traffic roofline for fp32 (108 GB/step at 800 GB/s) is 135 ms/step, i.e.
~118 tok/s aggregate at batch-16 if overhead truly vanishes. This notebook
measures how close we get.
"""


In [ ]:
import json
import os
import time

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.95")

import jax
import jax.numpy as jnp
import numpy as np

print(f"jax {jax.__version__} | backend {jax.default_backend()}")

GGUF = os.environ.get(
    "GGUF_PATH", "/content/gguf/Ternary-Bonsai-2-27B-PTQ1_0.gguf")
N_GEN = int(os.environ.get("N_TOKENS", "8"))
PROMPT = os.environ.get("PROMPT", "Hello")
BATCH = int(os.environ.get("BATCH", "16"))
MAX_T = int(os.environ.get("MAX_T", "256"))


In [ ]:
_src = open(os.environ.get("FWD_SRC", "/content/04.py"),
            encoding="utf-8").read()
fw = {"__name__": "fwd"}
exec(compile(_src.split("st = {")[0], "fwd.py", "exec"), fw)
_src = open(os.environ.get("FWD_SRC", "/content/04.py"),
            encoding="utf-8").read()
fw = {"__name__": "fwd"}
exec(compile(_src.split("st = {")[0], "fwd.py", "exec"), fw)
print("base loaded")

cfg = fw["cfg"]
NL, IS_RECR = cfg["n_layer"], tuple(cfg["is_recr"])
EPS, NEMB = cfg["eps"], cfg["n_embd"]
H, HKV, HD = cfg["n_head"], cfg["n_head_kv"], cfg["head_dim"]
ST, NV, NK, REP = cfg["state"], cfg["n_v"], cfg["n_k"], cfg["rep"]
KEY_DIM, CONV, CONV_DIM = cfg["key_dim"], cfg["conv"], cfg["conv_dim"]
NFF = cfg["n_ff"]

rms, rope, tok = fw["rms"], fw["rope"], fw["tok"]
l2n, fold = fw["l2n"], fw["fold"]
decode_ptq1_dev = fw["decode_ptq1_dev"]
embed, O_NORM = fw["embed"], fw["O_NORM"]
chunked_logits = fw["chunked_logits"]
SIGNS, ROW = fw["SIGNS"], fw["SIGNS"][NEMB]
LAYER_RAW = fw["LAYER_RAW"]
LAYER_MAP = fw["LAYER_MAP"]
SMALLW = fw["SMALLW"]
IS_ARR = jnp.asarray(IS_RECR)

_idxtri = jnp.arange(NV) % NK
_hk = jnp.arange(NK)[:, None]
_r = jnp.arange(REP)[None, :]
_perm = _hk + _r * NK
_kidx = jnp.arange(H) // (H // HKV)
_isscale = 1.0 / jnp.sqrt(jnp.float32(HD))
_sscale = 1.0 / jnp.sqrt(jnp.float32(ST))


@jax.jit
def gdn_unit(x, W, S, SW, conv_hist):
    xf = fold(x, ROW)
    qkv = xf @ W["attn_qkv.weight"]
    z = xf @ W["attn_gate.weight"]
    beta = jax.nn.sigmoid(x @ SW["ssm_beta.weight"])
    alpha = x @ SW["ssm_alpha.weight"]
    gate = SW["ssm_a"] * jnp.logaddexp(alpha + SW["ssm_dt.bias"], 0.0)
    win = jnp.concatenate([conv_hist, qkv[None, :]], 0)
    y = jnp.sum(win * SW["ssm_conv1d.weight"], 0)
    y = y * jax.nn.sigmoid(y)
    qk = l2n(y[:2 * KEY_DIM].reshape(2 * NK, ST).T, EPS)
    q, k = qk[:, :NK], qk[:, NK:]
    v = y[2 * KEY_DIM:].reshape(NV, ST).T
    q, k = q[:, _idxtri], k[:, _idxtri]
    Sg = S * jnp.exp(gate)[None, None, :]
    sk = jnp.einsum("ijv,iv->jv", Sg, k)
    dv = (v - sk) * beta[None, :]
    Sg = Sg + jnp.einsum("iv,jv->ijv", k, dv)
    o = jnp.einsum("ijv,iv->jv", Sg, q) * _sscale
    zz = z.reshape(NV, ST).T
    o = o * jax.lax.rsqrt(jnp.mean(o * o, axis=0, keepdims=True) + EPS)
    o = o * SW["ssm_norm.weight"][:, None]
    o = jax.nn.sigmoid(zz) * zz * o
    Q = jnp.transpose(o[:, _perm], (1, 2, 0))
    out = fold(Q.reshape(-1), SIGNS[ST * NV]) @ W["ssm_out.weight"]
    return out, Sg, win[1:, :]


@jax.jit
def attn_unit(x, W, SW, K, V, pos):
    xf = fold(x, ROW)
    qg = (xf @ W["attn_q.weight"]).reshape(H, 2, HD)
    q, g = qg[:, 0, :], qg[:, 1, :]
    q = rms(q, SW["attn_q_norm.weight"], EPS)
    k = rms((xf @ W["attn_k.weight"]).reshape(HKV, HD),
            SW["attn_k_norm.weight"], EPS)
    v = (xf @ W["attn_v.weight"]).reshape(HKV, HD)
    q, k = rope(q, pos), rope(k, pos)
    K = jax.lax.dynamic_update_slice(K, k[None, :, :], (pos, 0, 0))
    V = jax.lax.dynamic_update_slice(V, v[None, :, :], (pos, 0, 0))
    scores = jnp.einsum("hd,thd->ht", q, K[:, _kidx, :]) * _isscale
    allow = jnp.arange(K.shape[0]) <= pos
    scores = jnp.where(allow[None, :], scores, -1e30)
    p = jax.nn.softmax(scores, -1)
    o = jnp.einsum("ht,thd->hd", p, V[:, _kidx, :])
    o = o * jax.nn.sigmoid(g)
    return fold(o.reshape(-1), SIGNS[H * HD]) @ W["attn_output.weight"], K, V


@jax.jit
def ffn_unit(x, W):
    xf = fold(x, ROW)
    g = xf @ W["ffn_gate.weight"]
    u = xf @ W["ffn_up.weight"]
    return fold(jax.nn.silu(g) * u, SIGNS[NFF]) @ W["ffn_down.weight"]


gdn_batch = jax.vmap(gdn_unit, in_axes=(0, None, 0, None, 0))
attn_batch = jax.vmap(attn_unit, in_axes=(0, None, None, 0, 0, None))
ffn_batch = jax.vmap(ffn_unit, in_axes=(0, None))
logits_batch = jax.vmap(lambda h: chunked_logits(rms(h, O_NORM, EPS)))


"""
## Stacked weights (one-time, static shapes for scan)

`RAW_ALL` holds every layer's ternary blob (attn padded to GDN length).
`SMALL` holds each small weight stacked over layers. `GMAP`/`AMAP` are the
static per-projection slices for GDN vs attention layers.
"""


In [ ]:
gdn0 = next(i for i in range(NL) if IS_RECR[i])
att0 = next(i for i in range(NL) if not IS_RECR[i])
MAXB = max(len(LAYER_RAW[i]) for i in range(NL))
print(f"GDN exemplar {gdn0}, attn exemplar {att0}, MAXB {MAXB}")
# relocate to host, free device, then stack (avoids 5.9+5.3 GB peak)
import gc
RAWS = [np.asarray(LAYER_RAW[i]) for i in range(NL)]
for i in range(NL):
    if len(RAWS[i]) < MAXB:
        RAWS[i] = np.concatenate(
            [RAWS[i], np.zeros(MAXB - len(RAWS[i]), np.uint8)])
del fw["LAYER_RAW"]
del LAYER_RAW
gc.collect()
RAW_ALL = jnp.asarray(np.stack(RAWS))
del RAWS
gc.collect()
print(f"RAW_ALL {RAW_ALL.shape} "
      f"({RAW_ALL.nbytes / 2**30:.2f} GiB)")
GKEYS = sorted(SMALLW[gdn0].keys())
AKEYS = sorted(SMALLW[att0].keys())
GIDX = [i for i in range(NL) if IS_RECR[i]]
AIDX = [i for i in range(NL) if not IS_RECR[i]]
GSMALL = {k: jnp.stack([SMALLW[i][k] for i in GIDX]) for k in GKEYS}
ASMALL = {k: jnp.stack([SMALLW[i][k] for i in AIDX]) for k in AKEYS}
print(f"GDN small {len(GSMALL)} keys x48, attn small {len(ASMALL)} keys x16")
GMAP, AMAP = LAYER_MAP[gdn0], LAYER_MAP[att0]
print(f"GDN map {len(GMAP)} proj, attn map {len(AMAP)} proj")


def decode_stacked(raw, mp):
    flat = decode_ptq1_dev(raw).reshape(-1)
    return {k: flat[o:o + n].reshape(dims, order="F")
            for k, (o, n, dims) in mp.items()}


"""
## Scan body: one layer inside the compiled program

`il` is traced; weights come from stacked arrays, the branch from
`IS_ARR[il]`, and state stacks advance via per-kind counters. The compute
calls notebook 06's batched units unchanged.
"""


In [ ]:
def scan_body(carry, il):
    x, Sst, Cst, Kst, Vst, gpos, apos, pos, RAW_ALL, GSMALL, ASMALL = carry
    raw = RAW_ALL[il]
    is_gdn = IS_ARR[il]

    def gdn_path(_):
        SW = {k: GSMALL[k][gpos] for k in GKEYS}
        h = rms(x, SW["attn_norm.weight"], EPS)
        W = decode_stacked(raw, GMAP)
        a, ns, nc = gdn_batch(h, W, Sst[gpos], SW, Cst[gpos])
        S2 = Sst.at[gpos].set(ns)
        C2 = Cst.at[gpos].set(nc)
        f = ffn_batch(rms(x + a, SW["post_attention_norm.weight"], EPS), W)
        return a, f, S2, C2, Kst, Vst, gpos + 1, apos, RAW_ALL, GSMALL, ASMALL

    def attn_path(_):
        SW = {k: ASMALL[k][apos] for k in AKEYS}
        h = rms(x, SW["attn_norm.weight"], EPS)
        W = decode_stacked(raw, AMAP)
        a, nk, nv = attn_batch(h, W, SW, Kst[apos], Vst[apos], pos)
        K2 = Kst.at[apos].set(nk)
        V2 = Vst.at[apos].set(nv)
        f = ffn_batch(rms(x + a, SW["post_attention_norm.weight"], EPS), W)
        return a, f, Sst, Cst, K2, V2, gpos, apos + 1, RAW_ALL, GSMALL, ASMALL

    a, f, Sst, Cst, Kst, Vst, gpos, apos, _, _, _ = jax.lax.cond(
        is_gdn, gdn_path, attn_path, None)
    x = x + a + f
    return (x, Sst, Cst, Kst, Vst, gpos, apos, pos,
            RAW_ALL, GSMALL, ASMALL), None


@jax.jit
def scan_step(x, Sst, Cst, Kst, Vst, pos, RAW_ALL, GSMALL, ASMALL):
    (x, Sst, Cst, Kst, Vst, _, _, _, _, _, _), _ = jax.lax.scan(
        scan_body, (x, Sst, Cst, Kst, Vst, jnp.int32(0), jnp.int32(0), pos,
                    RAW_ALL, GSMALL, ASMALL),
        jnp.arange(NL))
    return x, Sst, Cst, Kst, Vst


"""
## Correctness gate: one scan step vs notebook 06's loop (bit-exact)

Both run fp32 with identical ops in identical order, so any divergence is
a scan-wiring bug, not numerics. Asserts before timing.
"""


In [ ]:
ids = tok.encode(PROMPT)
B = BATCH
print(f"prompt {PROMPT!r} -> {len(ids)} tokens | batch {B} | MAX_T {MAX_T}")


def new_state():
    Sst = jnp.zeros([48, B, ST, ST, NV], jnp.float32)
    Cst = jnp.zeros([48, B, CONV - 1, CONV_DIM], jnp.float32)
    Kst = jnp.zeros([16, B, MAX_T, HKV, HD], jnp.float32)
    Vst = jnp.zeros([16, B, MAX_T, HKV, HD], jnp.float32)
    return Sst, Cst, Kst, Vst


Sst, Cst, Kst, Vst = new_state()
x = jnp.broadcast_to(embed(int(ids[0])), (B, NEMB))
t0 = time.perf_counter()
x, Sst, Cst, Kst, Vst = scan_step(x, Sst, Cst, Kst, Vst, jnp.int32(0), RAW_ALL, GSMALL, ASMALL)
jax.block_until_ready(x)
print(f"scan warmup (compile+prefill): {time.perf_counter()-t0:.1f}s")
first = np.asarray(jnp.argmax(logits_batch(x), axis=-1))
print(f"first token: {int(first[0])} {tok.decode([int(first[0])])!r} "
      f"(expect 11 ',')")
assert int(first[0]) == 11, "scan diverged from the proven path"
print("gate PASS")


In [ ]:
gen = [[] for _ in range(B)]
cur = first
pos = jnp.int32(1)
t0 = time.perf_counter()
for i_step in range(N_GEN):
    for b in range(B):
        if cur[b] != tok.eos:
            gen[b].append(int(cur[b]))
    x = jnp.stack([embed(int(c)) for c in cur])
    x, Sst, Cst, Kst, Vst = scan_step(x, Sst, Cst, Kst, Vst, pos, RAW_ALL, GSMALL, ASMALL)
    cur = np.asarray(jnp.argmax(logits_batch(x), axis=-1))
    pos = pos + 1
    if i_step % 4 == 0:
        print(f"  step {i_step}: {tok.decode([int(cur[0])])!r}")
t_gen = time.perf_counter() - t0

n = sum(len(g) for g in gen)
agg = B * N_GEN / t_gen
print("\n=== notebook 08 receipt ===")
print(f"batch            : {B} (one prompt replicated)")
print(f"tokens           : {n} of {B * N_GEN} (eos-truncated)")
print(f"decode wall time : {t_gen:.2f}s")
print(f"aggregate tok/s  : {agg:.1f}")
print(f"per-seq latency  : {t_gen / max(N_GEN, 1) * 1000:.0f} ms/token")
print(f"baseline (nb 04) : 0.27 tok/s single-stream")
print(f"notebook 06      : 3.5 tok/s aggregate (batch-16)")
print(f"sample: {tok.decode(gen[0][:48])!r}")

receipt = {
    "notebook": "08-lax-scan-over-layers",
    "backend": str(jax.default_backend()),
    "jax": jax.__version__,
    "batch": B,
    "max_t": MAX_T,
    "prompt": PROMPT,
    "tokens_generated": n,
    "tokens_possible": B * N_GEN,
    "decode_seconds": round(t_gen, 3),
    "aggregate_tok_per_s": round(agg, 2),
    "per_sequence_ms_per_token": round(t_gen / max(N_GEN, 1) * 1000, 1),
    "baseline_notebook04_tok_per_s": 0.27,
    "notebook06_tok_per_s": 3.5,
    "correctness_gate_pass": True,
    "claim": "[exec] measured on Colab TPU v5e-1; batch replicates one prompt",
}
path = os.environ.get("RECEIPT", "/content/receipt-colab-tpu-08.json")
json.dump(receipt, open(path, "w"), indent=2)
print(f"receipt -> {path}")
